# Fase 1: Módulo NLP y Lógica de Priorización
## Sistema de Triaje Multimodal Clínico (NLP + Visión por Computadora)

Este notebook implementa el pipeline completo de Procesamiento de Lenguaje Natural (NLP) para el triaje clínico:
1. **Carga y análisis del corpus sintético clínico en español** (`tickets_nlp.csv`, 180 registros balanceados).
2. **Clasificador de Intención:** `TfidfVectorizer` + `CalibratedClassifierCV(LinearSVC)` para obtener probabilidades bien calibradas con detección de `INTENCION_AMBIGUA` (< 0.65).
3. **Clasificador de Sentimiento:** `TfidfVectorizer` + `LogisticRegression` (`NEGATIVO`, `NEUTRAL`, `POSITIVO`).
4. **Extractor de Entidades Híbrido (NER):** Expresiones regulares + normalización sin tildes para extraer `SEDE`, `EQUIPO_O_INSUMO`, `TIPO_DOCUMENTO` y `URGENCIA`.
5. **Serialización del artefacto:** Exportación hacia `backend/models/nlp_bundle.joblib`.
6. **Batería de pruebas de inferencia:** Evaluación de 5 casos clínicos reales (incluyendo caso ambiguo).

In [ ]:
# Instalación de dependencias (para Google Colab)
!pip install --quiet scikit-learn pandas joblib

In [ ]:
import os
import re
import json
import unicodedata
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Entorno inicializado correctamente.")

### 1. Carga del Corpus Sintético Clínico

In [ ]:
dataset_path = "datasets/tickets_nlp.csv"
if not os.path.exists(dataset_path):
    dataset_path = "training/datasets/tickets_nlp.csv"

df = pd.read_csv(dataset_path, encoding="utf-8")
print(f"Total registros: {len(df)}")
print("\nDistribución por Intención:")
print(df["intencion"].value_counts())
print("\nDistribución por Sentimiento:")
print(df["sentimiento"].value_counts())
print("\nDistribución por Urgencia:")
print(df["urgencia"].value_counts())
df.head()

### 2. División de Datos (80% Entrenamiento / 20% Validación Estratificada)

In [ ]:
X = df["texto"].values
y_intent = df["intencion"].values
y_sent = df["sentimiento"].values

X_train, X_test, y_intent_train, y_intent_test, y_sent_train, y_sent_test = train_test_split(
    X, y_intent, y_sent,
    test_size=0.20,
    random_state=42,
    stratify=y_intent
)

print(f"Muestras Train: {len(X_train)} | Muestras Test: {len(X_test)}")

### 3. Clasificador de Intención (TF-IDF + Calibrated LinearSVC)

In [ ]:
intent_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True
)
X_train_intent_vec = intent_vectorizer.fit_transform(X_train)
X_test_intent_vec = intent_vectorizer.transform(X_test)

base_svc = LinearSVC(class_weight="balanced", random_state=42, dual="auto")
intent_classifier = CalibratedClassifierCV(estimator=base_svc, cv=5)
intent_classifier.fit(X_train_intent_vec, y_intent_train)

y_intent_pred = intent_classifier.predict(X_test_intent_vec)
acc_intent = accuracy_score(y_intent_test, y_intent_pred)

print(f"Accuracy de Intención: {acc_intent * 100:.2f}%\n")
print(classification_report(y_intent_test, y_intent_pred, digits=4))
print("Matriz de Confusión:")
print(confusion_matrix(y_intent_test, y_intent_pred))

### 4. Clasificador de Sentimiento (TF-IDF + Logistic Regression)

In [ ]:
sentiment_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True
)
X_train_sent_vec = sentiment_vectorizer.fit_transform(X_train)
X_test_sent_vec = sentiment_vectorizer.transform(X_test)

sentiment_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
sentiment_classifier.fit(X_train_sent_vec, y_sent_train)

y_sent_pred = sentiment_classifier.predict(X_test_sent_vec)
acc_sent = accuracy_score(y_sent_test, y_sent_pred)

print(f"Accuracy de Sentimiento: {acc_sent * 100:.2f}%\n")
print(classification_report(y_sent_test, y_sent_pred, digits=4))
print("Matriz de Confusión:")
print(confusion_matrix(y_sent_test, y_sent_pred))

### 5. Extractor de Entidades Híbrido (Hybrid NER)

In [ ]:
class HybridNER:
    @staticmethod
    def normalize(text: str) -> str:
        if not text:
            return ""
        text = text.lower()
        return "".join(
            c for c in unicodedata.normalize("NFD", text)
            if unicodedata.category(c) != "Mn"
        )

    def __init__(self):
        self.sedes = [
            ("Sede Central", [r"\bsede central\b", r"\bcentral\b"]),
            ("Sede Norte", [r"\bsede norte\b", r"\bnorte\b"]),
            ("Sede Sur", [r"\bsede sur\b", r"\bsur\b"]),
            ("Laboratorio Principal", [r"\blaboratorio principal\b", r"\blab principal\b", r"\blaboratorio\b", r"\blab\b"]),
            ("Consultorio 1", [r"\bconsultorio 1\b", r"\bcons 1\b", r"\bconsultorio uno\b"]),
            ("Consultorio 2", [r"\bconsultorio 2\b", r"\bcons 2\b", r"\bconsultorio dos\b"]),
            ("Consultorio 3", [r"\bconsultorio 3\b", r"\bcons 3\b", r"\bconsultorio tres\b"]),
        ]
        self.equipos = [
            ("impresora térmica", [r"\bimpresora termica\b", r"\bimp\.?\s*termica\b", r"\bimpresora\b", r"\bcabezal termico\b", r"\brodillo\b"]),
            ("lector de código de barras", [r"\blector de codigo de barras\b", r"\blector de barras\b", r"\blector\b", r"\bescaner\b", r"\bscanner\b"]),
            ("analizador hematológico", [r"\banalizador hematologico\b", r"\banalizador hematologia\b", r"\banalizador\b"]),
            ("centrífuga", [r"\bcentrifuga\b", r"\bcentrifugadora\b"]),
            ("etiqueta", [r"\betiqueta\b", r"\betiquetas\b", r"\bsticker\b"]),
            ("tubo de muestra", [r"\btubo de muestra\b", r"\btubo\b", r"\btubos\b", r"\bfrasco\b"]),
            ("tóner", [r"\btoner\b", r"\btinta\b", r"\bcartucho\b"]),
            ("monitor", [r"\bmonitor\b", r"\bpantalla\b"]),
        ]
        self.documentos = [
            ("orden médica", [r"\borden medica\b", r"\bord\.?\s*med\.?\b", r"\borden med\b", r"\borden\b", r"\breceta\b"]),
            ("comprobante de pago", [r"\bcomprobante de pago\b", r"\bcomprobante\b", r"\bcomprovante\b", r"\bvoucher\b"]),
            ("boleta", [r"\bboleta electronica\b", r"\bboleta\b", r"\bfactura\b"]),
            ("DNI", [r"\bdni\b", r"\bdocumento de identidad\b"]),
            ("resultado de laboratorio", [r"\bresultado de laboratorio\b", r"\bresultado lab\b", r"\banalisis de sangre\b"]),
            ("consentimiento informado", [r"\bconsentimiento informado\b", r"\bconsentimiento\b"]),
        ]
        self.urg_alta = [r"\burgente\b", r"\burgnte\b", r"\binmediato\b", r"\bemergencia\b", r"\bcritico\b", r"\bhoy\b", r"\bbloqueado\b", r"\bpaciente esperando\b", r"\bdolor agudo\b"]
        self.urg_baja = [r"\bconsulta\b", r"\binformes\b", r"\bhorario\b", r"\bprecio\b", r"\bgracias\b", r"\bconfirmar\b", r"\bduda\b", r"\bcuando\b"]
        self.urg_media = [r"\bpronto\b", r"\bayuda\b", r"\brevision\b", r"\bproblema\b", r"\bfalla\b", r"\breprogramar\b", r"\bdemora\b", r"\bborroso\b", r"\bboroso\b"]

    def extract_entities(self, text: str) -> dict:
        nt = self.normalize(text)
        sede = "NONE"
        for c, pats in self.sedes:
            if any(re.search(p, nt) for p in pats): sede = c; break
        equipo = "NONE"
        for c, pats in self.equipos:
            if any(re.search(p, nt) for p in pats): equipo = c; break
        doc = "NONE"
        for c, pats in self.documentos:
            if any(re.search(p, nt) for p in pats): doc = c; break
        urg = "MEDIA"
        if any(re.search(p, nt) for p in self.urg_alta): urg = "ALTA"
        elif any(re.search(p, nt) for p in self.urg_baja): urg = "BAJA"
        elif any(re.search(p, nt) for p in self.urg_media): urg = "MEDIA"
        return {"sede": sede, "equipo_insumo": equipo, "tipo_documento": doc, "urgencia": urg}

ner = HybridNER()
print("NER Híbrido inicializado.")

### 6. Empaquetado en `nlp_bundle.joblib`

In [ ]:
bundle_data = {
    "intent_vectorizer": intent_vectorizer,
    "intent_classifier": intent_classifier,
    "sentiment_vectorizer": sentiment_vectorizer,
    "sentiment_classifier": sentiment_classifier,
    "intent_classes": list(intent_classifier.classes_),
    "sentiment_classes": list(sentiment_classifier.classes_),
    "nlp_confidence_threshold": 0.65
}

bundle_path = "../backend/models/nlp_bundle.joblib"
if not os.path.exists("../backend/models"):
    bundle_path = "nlp_bundle.joblib"

joblib.dump(bundle_data, bundle_path)
print(f"Bundle guardado en: {bundle_path}")

### 7. Batería de 5 Casos de Prueba de Inferencia

In [ ]:
casos = [
    ("Caso 1 (Falla Crítica)", "Urgente: La impresora térmica de Sede Sur tiene atasco de papel y no imprime nada, cabezal dañado"),
    ("Caso 2 (Doc Borroso)", "El paciente envió foto de orden médica muy borosa y cortada en los bordes, no se lee"),
    ("Caso 3 (Gestión Cita)", "Buenas tardes, solicito por favor reprogramar mi cita médica para la próxima semana"),
    ("Caso 4 (Consulta Ayuno)", "Buenos días, ¿cuáles son los requisitos de ayuno para el análisis de glucosa en el laboratorio?"),
    ("Caso 5 (Ambiguo/Vago)", "hola ayuda con esto porfa no pasa")
]

for titulo, texto in casos:
    x_i = intent_vectorizer.transform([texto])
    probs = intent_classifier.predict_proba(x_i)[0]
    top_idx = probs.argmax()
    top_prob = probs[top_idx]
    raw_intent = intent_classifier.classes_[top_idx]
    final_intent = "INTENCION_AMBIGUA" if top_prob < 0.65 else raw_intent
    
    x_s = sentiment_vectorizer.transform([texto])
    sent = sentiment_classifier.predict(x_s)[0]
    ents = ner.extract_entities(texto)
    
    print(f"=== {titulo} ===")
    print(f"Texto: {texto}")
    print(f"-> Intención: {final_intent} (Confianza: {top_prob:.4f}) | Base: {raw_intent}")
    print(f"-> Sentimiento: {sent}")
    print(f"-> Entidades: {ents}\n")